# 🤖 Model Comparison Tool - Compare AI Models Side by Side

**Compare different Hugging Face models on the same task**

![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)

---

## What This Does

- \ud83d\udd0c Compare multiple models on the same prompt
- \ud83d\udcc8 Benchmark generation speed
- \ud83d\udca1 Evaluate response quality
- \ud83d\udcb0 Memory usage comparison

---


In [ ]:
# Configuration

# Models to compare (add as many as you want!)
MODELS_TO_COMPARE = [
    "gpt2",
    "distilbert/distilgpt2",
    "microsoft/DialoGPT-medium",
]

# Test prompt
TEST_PROMPT = "Explain quantum computing in simple terms:"

# Generation settings
MAX_NEW_TOKENS = 100
TEMPERATURE = 0.7

print(f"\ud83d\udc49 Comparing {len(MODELS_TO_COMPARE)} models")
print(f"\ud83d\udc49 Prompt: {TEST_PROMPT[:50]}...")

## Step 1: Setup

In [ ]:
import os

# Security
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HOME"] = "/content/.hf_cache"
!mkdir -p /content/.hf_cache

# Install
!pip install -q transformers accelerate

print("\u2705 Setup complete!")

## Step 2: Define Comparison Functions

In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_model_for_comparison(model_id):
    """Load a model for comparison."""
    print(f"Loading {model_id}...")
    
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        
        # Add padding token if missing
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto" if torch.cuda.is_available() else "cpu"
        )
        model.eval()
        
        return {
            "status": "success",
            "model_id": model_id,
            "tokenizer": tokenizer,
            "model": model,
            "params": model.num_parameters(),
            "device": next(model.parameters()).device
        }
    except Exception as e:
        return {
            "status": "error",
            "model_id": model_id,
            "error": str(e)
        }

def generate_with_timing(model_info, prompt, max_tokens=100, temperature=0.7):
    """Generate text and measure time."""
    if model_info["status"] != "success":
        return {"error": model_info["error"]}
    
    tokenizer = model_info["tokenizer"]
    model = model_info["model"]
    device = model_info["device"]
    
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    # Time generation
    start_time = time.time()
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )
    
    end_time = time.time()
    generation_time = end_time - start_time
    
    # Decode
    generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    # Calculate tokens per second
    num_tokens = len(outputs[0]) - inputs.input_ids.shape[1]
    tokens_per_second = num_tokens / generation_time if generation_time > 0 else 0
    
    return {
        "text": generated_text,
        "time_seconds": generation_time,
        "tokens_generated": num_tokens,
        "tokens_per_second": tokens_per_second
    }

print("\u2705 Comparison functions ready!")

## Step 3: Run Comparison

In [ ]:
print("=" * 70)
print("MODEL COMPARISON")
print("=" * 70)
print(f"\nPrompt: {TEST_PROMPT}\n")

results = []

for model_id in MODELS_TO_COMPARE:
    print(f"\n{'─' * 70}")
    
    # Load model
    model_info = load_model_for_comparison(model_id)
    
    if model_info["status"] == "error":
        print(f"\u274c Error: {model_info['error']}")
        results.append({"model": model_id, "error": model_info["error"]})
        continue
    
    print(f"\u2705 Loaded! Parameters: {model_info['params']:,}")
    
    # Generate
    print("Generating...")
    gen_result = generate_with_timing(
        model_info, 
        TEST_PROMPT, 
        max_tokens=MAX_NEW_TOKENS, 
        temperature=TEMPERATURE
    )
    
    if "error" in gen_result:
        print(f"\u274c Generation error: {gen_result['error']}")
        continue
    
    # Store result
    results.append({
        "model": model_id,
        "params": model_info["params"],
        "text": gen_result["text"],
        "time": gen_result["time_seconds"],
        "tokens_per_sec": gen_result["tokens_per_second"]
    })
    
    print(f"\n\ud83d\udd04 Time: {gen_result['time_seconds']:.2f}s")
    print(f"\ud83d\udcca Speed: {gen_result['tokens_per_second']:.1f} tokens/sec")
    print(f"\ud83d\udcac Response:\n{gen_result['text']}")

## Step 4: Summary Table

In [ ]:
# Create summary table
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70 + "\n")

print(f"{'Model':<40} {'Params':>12} {'Time':>8} {'Tokens/s':>10}")
print("-" * 70)

for r in results:
    if "error" in r:
        print(f"{r['model']:<40} {'ERROR':>12}")
    else:
        print(f"{r['model']:<40} {r['params']/1e6:>10.1f}M {r['time']:>7.2f}s {r['tokens_per_sec']:>9.1f}")

# Find best performers
valid_results = [r for r in results if "error" not in r]
if valid_results:
    fastest = min(valid_results, key=lambda x: x["time"])
    most_tokens = max(valid_results, key=lambda x: x["tokens_per_sec"])
    
    print("\n\ud83c\udfc6 Best Performance:")
    print(f"   Fastest: {fastest['model']} ({fastest['time']:.2f}s)")
    print(f"   Highest throughput: {most_tokens['model']} ({most_tokens['tokens_per_sec']:.1f} tokens/s)")

## Step 5: Memory Cleanup

In [ ]:
import gc

# Clean up models
del model, tokenizer, model_info, gen_result

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\u2705 Memory cleared!")

---

## \ud83d\udcca How to Interpret Results

| Metric | What It Means |
|--------|---------------|
| **Parameters** | Model size (more = generally better quality, slower) |
| **Time** | Total generation time (lower = faster) |
| **Tokens/sec** | Throughput (higher = more efficient) |

## Tips

- Smaller models (GPT2, DistilGPT2) are faster but less capable
- Compare models on tasks that matter for your use case
- Consider both speed AND quality for production

---

**Star**: [GitHub](https://github.com/unn-Known1/huggingface-colab-secure-setup)